# Lecture 5: Intro to dynamic programming; policy function iteration

**ECU44291 Quantitative Macroeconomics.**

Lecture 5 was mostly theory. This notebook provides the numbers behind the slides: every quantity quoted there is produced by a cell here, so you can check the board against the code. I'll walk through this so if you're reading right after that lecture don't stress if you don't understand it fully yet, but it's a useful exercise to try to map the code to that theory. 

Two models appear. The **life-cycle household** (ages, income, a borrowing limit), where the Euler equation is derived and tested along a simulated life. The **growth model** with log utility and full depreciation, $V(k) = \max_{k'} \ln(k^{\alpha} - k') + \beta V(k')$, with the closed form $V(k) = A + B \ln k$ and $g(k) = \alpha\beta k^{\alpha}$ as the check for the operator, the iteration count, the Euler-equation error, and the two ways of reusing the policy: modified policy iteration and policy function iteration.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import time

alpha, beta = 0.3, 0.9
k = np.linspace(0.02, 0.5, 200)
n = len(k)
max_iter, tol = 10000, 1e-6

def closed_form_value(k):
    B = alpha / (1 - alpha * beta)
    A = (1 / (1 - beta)) * ((alpha * beta / (1 - alpha * beta)) * np.log(alpha * beta) + np.log(1 - alpha * beta))
    return A + B * np.log(k)

def closed_form_policy(k):
    return alpha * beta * k ** alpha

c = k[:, None] ** alpha - k[None, :]          # every (k_i, k_j) pair, once
U = np.full(c.shape, -np.inf)
U[c > 0] = np.log(c[c > 0])

def bellman(V, beta=beta):
    value = U + beta * V[None, :]
    return value.max(axis=1), value.argmax(axis=1)

## 1. The household's Euler equation, in numbers

The life-cycle household solved on a grid by backward induction, exactly as in the backward-induction lecture: sixty ages, forty-five of them working, $\sigma = 2$, $\beta = 0.96$, $R = 1.04$, a pension of 0.4 of average earnings, 300 asset points on $[0, 25]$, no borrowing.

The Euler equation says $u'(c_j) = \beta R\, u'(c_{j+1})$ at any age where the borrowing limit does not bind, and $u'(c_j) > \beta R\, u'(c_{j+1})$ where it does. With $u'(c) = c^{-\sigma}$ the unconstrained ratio is $c_{j+1}/c_j = (\beta R)^{1/\sigma}$.

NB: this is the Lecture 3 household. If you go back to that code you will be able to **see** the pattern we're talking about in that figure! 

**Question before running:** at age 21 the household has no assets and rising earnings ahead. What sign should $u'(c_j) - \beta R\, u'(c_{j+1})$ have?

In [ ]:
J, J_R, beta_h, R, sigma, repl, a_max, n_a = 60, 45, 0.96, 1.04, 2.0, 0.4, 25.0, 300
jj = np.arange(1, J + 1)
e = np.exp(0.06 * jj - 0.0012 * jj ** 2); e = e / e[:J_R].mean(); e[J_R:] = 0.0   # age-efficiency profile
y = e.copy(); y[J_R:] = repl * e[:J_R].mean()                                      # income: earnings, then the pension
a = np.linspace(0.0, a_max, n_a)

V_h = np.zeros((J + 1, n_a))                       # V[J] = 0: the terminal condition
policy_h = np.zeros((J, n_a), dtype=int)
for j in range(J - 1, -1, -1):                     # last age first
    cons = R * a[:, None] + y[j] - a[None, :]      # rows: assets today, columns: assets tomorrow
    u = np.full(cons.shape, -np.inf)
    u[cons > 0] = cons[cons > 0] ** (1 - sigma) / (1 - sigma)
    value = u + beta_h * V_h[j + 1][None, :]
    policy_h[j] = value.argmax(axis=1)
    V_h[j] = value.max(axis=1)

idx, assets, c_path = 0, [0.0], []                 # simulate a life from zero assets
for j in range(J):
    nxt = policy_h[j, idx]
    c_path.append(R * a[idx] + y[j] - a[nxt]); assets.append(a[nxt]); idx = nxt
c_path, assets = np.array(c_path), np.array(assets)

mu = c_path ** (-sigma)
residual = mu[:-1] - beta_h * R * mu[1:]           # u'(c_j) - beta R u'(c_{j+1})
print("unconstrained ratio c_{j+1}/c_j = (beta R)^(1/sigma) =", round((beta_h * R) ** (1 / sigma), 4))
for j in [0, 1, 2, 3, 4, 20, 30, 40]:
    print(f"age {21 + j}: assets next year {assets[j + 1]:5.2f}   c = {c_path[j]:.3f}   "
          f"u'(c_j) - beta R u'(c_j+1) = {residual[j]:+.4f}")

unconstrained ratio c_{j+1}/c_j = (beta R)^(1/sigma) = 0.9992
age 21: assets next year  0.00   c = 0.606   u'(c_j) - beta R u'(c_j+1) = +0.2946
age 22: assets next year  0.00   c = 0.641   u'(c_j) - beta R u'(c_j+1) = +0.2528
age 23: assets next year  0.00   c = 0.676   u'(c_j) - beta R u'(c_j+1) = +0.2175
age 24: assets next year  0.00   c = 0.712   u'(c_j) - beta R u'(c_j+1) = +0.1876
age 25: assets next year  0.00   c = 0.748   u'(c_j) - beta R u'(c_j+1) = +0.1622
age 41: assets next year  1.00   c = 0.965   u'(c_j) - beta R u'(c_j+1) = +0.0448
age 51: assets next year  3.93   c = 0.967   u'(c_j) - beta R u'(c_j+1) = -0.0084
age 61: assets next year  6.27   c = 0.966   u'(c_j) - beta R u'(c_j+1) = +0.1153


Positive, and large, at ages 21 to 25: the household is at the corner $a' = 0$, consuming its income, and would borrow if it could. Near zero from the mid-forties on, where it is saving: the Euler equation holds, up to the grid (choices are grid points, so the residual is not exactly zero). The sign is the inequality on the slide.

## 2. Blackwell's two conditions, checked

The Bellman operator $T$ for the growth model, on the grid, is `bellman`. Blackwell's conditions: (1) if $V \le W$ everywhere then $TV \le TW$ everywhere; (2) $T(V + c) = TV + \beta c$ for a constant $c$. Together they give $\|TV - TW\| \le \beta \|V - W\|$.

**Before running:** the discounting check prints a difference; how small should it be?

In [ ]:
# SOLUTION
rng = np.random.default_rng(0)
V = rng.normal(size=n)
W = V + np.abs(rng.normal(size=n))                # W >= V at every grid point
TV, _ = bellman(V)
TW, _ = bellman(W)
TVc, _ = bellman(V + 3.0)
print("monotone  (TV <= TW everywhere):      ", bool(np.all(TW >= TV - 1e-12)))
print("discounting |T(V+3) - (TV + beta*3)|:  ", np.max(np.abs(TVc - (TV + beta * 3.0))))
print("contraction ratio ||TV-TW|| / ||V-W||: ", round(np.max(np.abs(TV - TW)) / np.max(np.abs(V - W)), 4), " (beta =", beta, ")")

monotone  (TV <= TW everywhere):       True
discounting |T(V+3) - (TV + beta*3)|:   8.881784197001252e-16
contraction ratio ||TV-TW|| / ||V-W||:  0.609  (beta = 0.9 )


## 3. What the iterates are

Value function iteration from $V_0 = 0$, keeping every iterate. Iteration $n$ solves the $n$-period problem, so its policy at $k = 0.15$ should save the finite-horizon share $x_n = \alpha\beta/(1 + \alpha\beta - x_{n-1})$ from $x = 0$: the table on the slide.

**Question before running:**  how many iterations, and what is the first distance?

In [ ]:
def vfi(beta=beta, tol=tol):
    V = np.zeros(n)
    history, dists = [V.copy()], []
    for it in range(max_iter):
        V_new, policy = bellman(V, beta)
        dist = np.max(np.abs(V_new - V))
        dists.append(dist)
        V = V_new                                  # a new array, not a rename
        history.append(V.copy())
        if dist < tol:
            return V, policy, np.array(dists), history
    raise RuntimeError("no convergence")

V, policy, dists, history = vfi()
print(f"{len(dists)} iterations; first distance {dists[0]:.3f}; max |V - closed form| = {np.max(np.abs(V - closed_form_value(k))):.4f}")
i = np.argmin(np.abs(k - 0.15))
x = 0.0
for m in range(1, 7):
    _, pol_m = bellman(history[m - 1])
    print(f"iteration {m}: VFI saving rate {k[pol_m[i]] / k[i] ** alpha:.3f}   {m} periods left {x:.3f}")
    x = alpha * beta / (1 + alpha * beta - x)

132 iterations; first distance 1.240; max |V - closed form| = 0.0001
iteration 1: VFI saving rate 0.035   1 periods left 0.000
iteration 2: VFI saving rate 0.218   2 periods left 0.213
iteration 3: VFI saving rate 0.257   3 periods left 0.255
iteration 4: VFI saving rate 0.265   4 periods left 0.266
iteration 5: VFI saving rate 0.270   5 periods left 0.269
iteration 6: VFI saving rate 0.270   6 periods left 0.270


## 4. The rate of convergence, and the iteration count

The ratio of consecutive distances settles at $\beta$. The number of iterations from a first distance $d_0$ to a tolerance is about $\ln(\text{tol}/d_0)/\ln\beta$. The slide works it for $d_0 = 10$ at $\beta = 0.95$ and $0.99$; the cell checks the bound against the actual count at $\beta = 0.9$ and $0.99$.

**Question before running:** our first distance is 1.24. Will the bound be above or below the actual 132?

In [5]:
print("late ratios:", np.round(dists[-5:] / dists[-6:-1], 4), " beta =", beta)
for b in (0.9, 0.99):
    _, _, d, _ = vfi(beta=b)
    bound = int(np.ceil(np.log(tol / d[0]) / np.log(b)))
    print(f"beta = {b}: bound {bound} iterations, actual {len(d)}")
print()
d0 = 10.0
for b in (0.95, 0.99):
    exact = np.log(tol / d0) / np.log(b)
    print(f"the slide's arithmetic at beta = {b}, d0 = 10: ln(1e-7)/ln({b}) = {np.log(tol/d0):.2f}/{np.log(b):.3f} = {exact:.1f}, so {int(np.ceil(exact))}")

late ratios: [0.9 0.9 0.9 0.9 0.9]  beta = 0.9
beta = 0.9: bound 134 iterations, actual 132
beta = 0.99: bound 1397 iterations, actual 1363

the slide's arithmetic at beta = 0.95, d0 = 10: ln(1e-7)/ln(0.95) = -16.12/-0.051 = 314.2, so 315
the slide's arithmetic at beta = 0.99, d0 = 10: ln(1e-7)/ln(0.99) = -16.12/-0.010 = 1603.7, so 1604


## 5. What a converged $V$ does not establish

The tolerance is on the change in $V$. The policy's accuracy is set by the grid. Two runs with tolerances $10^{-4}$ and $10^{-6}$ give value functions that differ by less than $10^{-3}$, and policies that differ at a few grid points by a whole grid step: the objective is flat there.

The check that measures the policy is the Euler-equation error: $\log_{10} |1 - c\,\beta\alpha k'^{\alpha-1}/c'|$ at grid midpoints, with the policy interpolated. $-2$ means a one-percent error in the consumption decision.

**Question before running:** the grid policy on 50 points scores about $-2.1$. Better or worse on 200 points, and by how much?

In [6]:
V4, pol4, d4, _ = vfi(tol=1e-4)
print(f"tol 1e-4 vs 1e-6: max |V difference| {np.max(np.abs(V4 - V)):.2e}; policy differs at {np.sum(pol4 != policy)} of {n} grid points")

def euler_errors(kgrid, g_values):
    km = 0.5 * (kgrid[:-1] + kgrid[1:])
    g = lambda x: np.interp(x, kgrid, g_values)
    kp = g(km); c0 = km ** alpha - kp; c1 = kp ** alpha - g(kp)
    return np.log10(np.abs(1 - c0 * beta * alpha * kp ** (alpha - 1) / c1))

print(f"grid policy, 200 points: mean log10 Euler error {euler_errors(k, k[policy]).mean():.2f}")
k50 = np.linspace(0.02, 0.5, 50)
c50 = k50[:, None] ** alpha - k50[None, :]; U50 = np.full(c50.shape, -np.inf); U50[c50 > 0] = np.log(c50[c50 > 0])
V50 = np.zeros(50)
for it in range(max_iter):
    val50 = U50 + beta * V50[None, :]; V50_new = val50.max(axis=1)
    if np.max(np.abs(V50_new - V50)) < tol: V50 = V50_new; break
    V50 = V50_new
print(f"grid policy,  50 points: mean log10 Euler error {euler_errors(k50, k50[val50.argmax(axis=1)]).mean():.2f}")

tol 1e-4 vs 1e-6: max |V difference| 8.46e-04; policy differs at 0 of 200 grid points
grid policy, 200 points: mean log10 Euler error -2.63
grid policy,  50 points: mean log10 Euler error -2.10


## 6. Reusing the policy: modified policy iteration

Value function iteration maximises 132 times, but the policy stops changing long before the values do. The maximisation is the expensive step: $N$ comparisons at every grid point. Modified policy iteration keeps the policy $\sigma$ that the last maximisation chose and, before maximising again, applies the Bellman update with that choice held fixed, $m$ times:
$$V \leftarrow r_\sigma + \beta\, V[\sigma], \qquad r_\sigma(k_i) = U[i, \sigma(i)].$$
There is no max in this update: one lookup per grid point. The stopping rule is still the tolerance on the change in $V$ across a maximisation. With $m = 0$ there are no updates, and the program is value function iteration.

**Question before running:** with $m = 0$, how many maximisations? How should the policy and $V$ compare with section 3's?

In [7]:
def modified_policy_iteration(m, beta=beta, tol=tol):
    rows = np.arange(n)
    V_m = np.zeros(n)
    for it in range(max_iter):
        V_new, pol = bellman(V_m, beta)                # one maximisation
        dist = np.max(np.abs(V_new - V_m))
        V_m = V_new
        if dist < tol:                                 # the stopping rule is still on V
            return V_m, pol, it + 1
        r = U[rows, pol]                               # the return under the policy just chosen
        for _ in range(m):                             # m updates with the policy held fixed: no max
            V_m = r + beta * V_m[pol]
    raise RuntimeError("no convergence")

V_m0, pol_m0, n_m0 = modified_policy_iteration(0)
print(f"m = 0: {n_m0} maximisations; same policy as value function iteration: {np.array_equal(pol_m0, policy)}; "
      f"same V, exactly: {np.array_equal(V_m0, V)}")

m = 0: 132 maximisations; same policy as value function iteration: True; same V, exactly: True


**Question before running:** predict the number of maximisations for $m = 1, 5, 20, 100$ at $\beta = 0.9$. Then at $\beta = 0.99$, where value function iteration takes 1,363.

In [8]:
for b in (0.9, 0.99):
    _, p_vfi, d_vfi, _ = vfi(beta=b)
    print(f"beta = {b}: value function iteration {len(d_vfi)} maximisations")
    for m in (0, 1, 5, 20, 100):
        t0 = time.perf_counter()
        _, p_m, n_m = modified_policy_iteration(m, beta=b)
        ms = 1000 * (time.perf_counter() - t0)
        print(f"  m = {m:3d}: {n_m:5d} maximisations, {ms:6.1f} ms, same policy {np.array_equal(p_m, p_vfi)}")

beta = 0.9: value function iteration 132 maximisations
  m =   0:   132 maximisations,    7.3 ms, same policy True
  m =   1:    66 maximisations,    3.6 ms, same policy True
  m =   5:    23 maximisations,    1.5 ms, same policy True
  m =  20:     8 maximisations,    0.8 ms, same policy True
  m = 100:     7 maximisations,    1.9 ms, same policy True


beta = 0.99: value function iteration 1363 maximisations

  m =   0:  1363 maximisations,   69.9 ms, same policy True


  m =   1:   682 maximisations,   37.1 ms, same policy True


  m =   5:   228 maximisations,   14.9 ms, same policy True
  m =  20:    66 maximisations,    6.5 ms, same policy True


  m = 100:    13 maximisations,    4.3 ms, same policy True


One update with the policy held fixed moves $V$ about as far as one maximisation, so $m = 1$ roughly halves the count. The count then falls fast and flattens: at $\beta = 0.9$ it does not go below 7, and from there the extra updates only cost time. The best $m$ is larger at high $\beta$, where there is more work for the updates to do.

The updates value a policy that is not yet optimal; the next maximisation corrects it. As $m$ grows, the updates converge to the exact value of following $\sigma$ forever, $v_\sigma = r_\sigma + \beta P_\sigma v_\sigma$. That is $N$ linear equations, which can be solved once instead. Solving them, and then stopping when the policy stops changing, is **policy function iteration**: the rest of this notebook. In code and in many texts both versions are called Howard's improvement, after Howard (1960); a linear solve means the exact version, a fixed number of updates the modified one.

## 7. Policy function iteration by hand: the three-size cake

Sizes $w \in \{0, 1, 2\}$, $u(c) = \sqrt{c}$, $\beta = 0.9$. Start from the policy that keeps nothing. Each step: build $r_\sigma$ and $P_\sigma$, solve $(I - \beta P_\sigma) v = r_\sigma$, improve, stop when the policy does not change. The board says two steps, ending at $v = (0, 1, 1.9)$.

**Question before running:** what is $v$ after evaluating "keep nothing"?

In [9]:
# SOLUTION
w = np.array([0, 1, 2])
U_cake = np.full((3, 3), -np.inf)
for i_ in range(3):
    for j_ in range(i_ + 1):
        U_cake[i_, j_] = np.sqrt(w[i_] - w[j_])

sigma_c = np.zeros(3, dtype=int)                  # keep nothing, everywhere
for step in range(1, 10):
    r = U_cake[np.arange(3), sigma_c]
    P = np.zeros((3, 3)); P[np.arange(3), sigma_c] = 1.0
    v = np.linalg.solve(np.eye(3) - 0.9 * P, r)
    improved = (U_cake + 0.9 * v[None, :]).argmax(axis=1)
    print(f"step {step}: policy {sigma_c}  r = {np.round(r, 3)}  v = {np.round(v, 3)}  improved policy {improved}")
    if np.array_equal(improved, sigma_c):
        print("policy unchanged: stop")
        break
    sigma_c = improved

step 1: policy [0 0 0]  r = [0.    1.    1.414]  v = [-0.     1.     1.414]  improved policy [0 0 1]
step 2: policy [0 0 1]  r = [0. 1. 1.]  v = [-0.   1.   1.9]  improved policy [0 0 1]
policy unchanged: stop


## 8. The code from the slide, run as printed

`policy_vfi` and `V_vfi` are the policy (grid indices) and the value function from section 3, run to convergence. Read each line before running it.

**Question before running:** what are the three printed values?

In [10]:
# SOLUTION
policy_vfi = policy
V_vfi = V

policy = np.zeros(n, dtype=int)
v = np.zeros(n)
for it in range(max_iter):
    r = U[np.arange(n), policy]
    P = np.zeros((n, n))
    P[np.arange(n), policy] = 1.0
    v = np.linalg.solve(np.eye(n) - beta * P, r)
    value = U + beta * v[None, :]
    new_policy = value.argmax(axis=1)
    if np.array_equal(new_policy, policy):
        break
    policy = new_policy
gap = round(np.max(np.abs(v - V_vfi)), 4)
print(it + 1, np.array_equal(policy, policy_vfi), gap)

6 True 0.0


Six steps; the same policy as value function iteration at all 200 grid points; the same value function to four decimals (the raw gap is about $10^{-5}$), because both are the value of the same optimal grid policy and value function iteration's remaining error is of the order of its tolerance. Against the closed form both sit 0.0001 away; that is the grid, not the method.

## 9. The count against value function iteration

**Workflow note.** *Measure before you optimise.* `%time` on a line runs it and reports how long it took. Four lines, both methods at both values of $\beta$, are the whole argument for policy function iteration at high $\beta$.

**Question before running:** value function iteration took 132 iterations at $\beta = 0.9$. How many at $\beta = 0.99$, and how many policy function iteration steps?

In [11]:
def policy_function_iteration(beta=beta):
    pol = np.zeros(n, dtype=int)
    for it in range(max_iter):
        r = U[np.arange(n), pol]
        P = np.zeros((n, n)); P[np.arange(n), pol] = 1.0
        v = np.linalg.solve(np.eye(n) - beta * P, r)
        new_pol = (U + beta * v[None, :]).argmax(axis=1)
        if np.array_equal(new_pol, pol):
            return v, pol, it + 1
        pol = new_pol
    raise RuntimeError("no convergence")

%time V9, p9, d9, _ = vfi(beta=0.9)
%time V99, p99, d99, _ = vfi(beta=0.99)
%time vH9, pH9, nH9 = policy_function_iteration(beta=0.9)
%time vH99, pH99, nH99 = policy_function_iteration(beta=0.99)
print(f"beta = 0.9:  VFI {len(d9)} iterations, PFI {nH9} steps, same policy {np.array_equal(p9, pH9)}")
print(f"beta = 0.99: VFI {len(d99)} iterations, PFI {nH99} steps, same policy {np.array_equal(p99, pH99)}")

CPU times: total: 15.6 ms
Wall time: 6.55 ms


CPU times: total: 62.5 ms
Wall time: 67.1 ms
CPU times: total: 0 ns
Wall time: 2.56 ms
CPU times: total: 15.6 ms
Wall time: 3.12 ms
beta = 0.9:  VFI 132 iterations, PFI 6 steps, same policy True
beta = 0.99: VFI 1363 iterations, PFI 8 steps, same policy True


**Question before running:** modified policy iteration with $m = 500$ evaluates each policy almost exactly. Does it need as many maximisations as policy function iteration takes steps?

In [12]:
for b in (0.9, 0.99):
    _, _, n500 = modified_policy_iteration(500, beta=b)
    _, _, nP = policy_function_iteration(beta=b)
    print(f"beta = {b}: modified policy iteration with m = 500: {n500} maximisations; policy function iteration: {nP} steps")

beta = 0.9: modified policy iteration with m = 500: 7 maximisations; policy function iteration: 6 steps
beta = 0.99: modified policy iteration with m = 500: 9 maximisations; policy function iteration: 8 steps


One more for the modified version. Its stopping rule is on $V$, so it needs one more maximisation to see that $V$ did not change. With the stopping rule on the policy, the step that returns the same policy already proves $v = Tv$. The two programs differ in two lines, the evaluation and the stopping rule, and the second change is allowed only because of the first.

## Before next week

- **Tutorial 2, Wednesday 21 October, 12:00, 5033 Arts:** thirty minutes walking through sections 6 to 9 of this notebook on your own laptop, from modified policy iteration to policy function iteration; then twenty minutes of M1 clinic on the grid solver and your two features. Bring your laptop and your solver.
- **The `m1-solo` tag is due Friday 23 October at 17:00** and covers the grid solver with two features, your tests and the handwritten derivation. The continuous-choice upgrade is Part C, in the study week, with an assistant and the log. The amended specification is on Blackboard.
- On paper, once: the household's Euler equation by both routes and the inequality at the borrowing limit; the contraction argument in three sentences; the iteration count at $\beta = 0.95$.
- Next lecture: the choice leaves the grid. Golden-section search, interpolation, and what goes wrong beyond the last grid point.